In [117]:
import refinitiv.data as rd
import pandas as pd
from datetime import datetime, timedelta
import random
import numpy as np


# Open the pre-configured CodeBook session
rd.open_session()

<refinitiv.data.session.Definition object at 0x7fa614d259a0 {name='codebook'}>

In [2]:
# Pull the list of current RICs in the JCI
index_constituents = rd.get_data(
    universe=["0#.JKSE"], 
    fields=["TR.RIC", "TR.CommonName"]
)

# 2. Clean and randomly sample 50 RICs
all_rics = [ric for ric in index_constituents["RIC"].unique() if ric and ric.strip()]
sample_size = 50
random_rics = random.sample(all_rics, min(sample_size, len(all_rics)))

# 3. Define 3-Year Window
end_date = datetime(2025, 12, 21)
start_date = end_date - timedelta(days=3*365)

print(f"Sampled {len(random_rics)} stocks for the period {start_date.date()} to {end_date.date()}")

Sampled 50 stocks for the period 2022-12-22 to 2025-12-21


In [88]:
# Updated fields to ensure historical population
daily_data = rd.get_history(
    universe=random_rics,
    fields=[
        "TR.PriceClose", 
        "TR.Volume", 
        "TR.Volatility10D"  # This is the historical timeseries version of volatility
    ],
    interval="1D",
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
    adjustments="EX"
)

/opt/conda/lib/python3.8/site-packages/refinitiv/data/_access_layer/get_history_func.py:137:UserWarning: 'adjustments' is not applicable for fields ['TR.PRICECLOSE', 'TR.VOLUME', 'TR.VOLATILITY10D']


In [89]:
print(daily_data.notnull().mean() * 100)

AMRT.JK  Price Close             70.068695
         Volume                  70.068695
         Volatility - 10 days     0.098135
MREI.JK  Price Close             70.068695
         Volume                  70.068695
                                   ...    
BYAN.JK  Volume                  70.068695
         Volatility - 10 days     0.098135
NICL.JK  Price Close             69.578018
         Volume                  70.068695
         Volatility - 10 days     0.098135
Length: 150, dtype: float64


In [90]:
# 1. Drop the rows that are entirely empty (the <NA> gaps)
daily_clean = daily_data.dropna(how='all')

# 2. Collapse duplicate dates by taking the first available value
# This removes the "double row" issue caused by adjustment parameters
daily_clean = daily_clean.groupby(level=0).first()

# 3. Stack the instruments so you have a clean panel
# This turns the columns (AMRT.JK, MREI.JK) into a single 'Instrument' column
daily_long = daily_clean.stack(level=0).reset_index()

# 4. Standardize Column Names
daily_long.columns = ['Date', 'Instrument', 'Price', 'Volume', 'Volatility_20D']

# 5. Sort by Instrument then Date for sequential analysis
daily_long = daily_long.sort_values(['Instrument', 'Date'])

print(f"Cleaned sequential data for {daily_long['Instrument'].nunique()} stocks.")
daily_long.head(10)

Cleaned sequential data for 50 stocks.


,Date,Instrument,Price,Volume,Volatility_20D
2,2022-12-22,ACST.JK,160,<NA>,1904600
52,2022-12-23,ACST.JK,158,<NA>,1857800
102,2022-12-26,ACST.JK,159,<NA>,3155200
152,2022-12-27,ACST.JK,160,<NA>,11489900
202,2022-12-28,ACST.JK,156,<NA>,3517200
252,2022-12-29,ACST.JK,159,<NA>,3604100
302,2022-12-30,ACST.JK,157,<NA>,3351900
352,2023-01-02,ACST.JK,160,<NA>,1370600
402,2023-01-03,ACST.JK,160,<NA>,12273000
452,2023-01-04,ACST.JK,158,<NA>,2339500


In [92]:
# 1. Clear out completely empty days (market holidays)
# This keeps the rows that have at least one piece of data
daily_clean = daily_data.dropna(how='all')

# 2. Handle the "Repeating Dates" issue correctly
# Instead of first(), use mean() or max() to ensure you grab the populated adjustment layer
daily_clean = daily_clean.groupby(level=0).first()

# 3. Stack the Tickers into a 'Long' format
# This moves AMRT.JK, MREI.JK, etc., into a column
daily_long = daily_clean.stack(level=0).reset_index()

# 4. SAFE COLUMN RENAMING
# Instead of daily_long.columns = [...], rename specifically to avoid shifting
daily_long = daily_long.rename(columns={
    'level_1': 'Instrument',
    'Price Close': 'Price',
    'Volume': 'Volume',
    'Volatility - 20 days': 'Volatility'
})

# 5. Final Sort for Thesis Sequential Analysis
# Essential for your Monday Effect returns calculation [cite: 81-84, 88]
daily_long = daily_long.sort_values(['Instrument', 'Date'])

print(f"Success! Volume preserved for {daily_long['Instrument'].nunique()} stocks.")
daily_long.head(10)

Success! Volume preserved for 50 stocks.


,Date,Instrument,Price,Volatility - 10 days,Volume
2,2022-12-22,ACST.JK,160,<NA>,1904600
52,2022-12-23,ACST.JK,158,<NA>,1857800
102,2022-12-26,ACST.JK,159,<NA>,3155200
152,2022-12-27,ACST.JK,160,<NA>,11489900
202,2022-12-28,ACST.JK,156,<NA>,3517200
252,2022-12-29,ACST.JK,159,<NA>,3604100
302,2022-12-30,ACST.JK,157,<NA>,3351900
352,2023-01-02,ACST.JK,160,<NA>,1370600
402,2023-01-03,ACST.JK,160,<NA>,12273000
452,2023-01-04,ACST.JK,158,<NA>,2339500


In [122]:
fundamentals = rd.get_data(
    universe=random_rics,
    fields=[
        "TR.CompanyMarketCapitalization.date",
        "TR.CompanyMarketCapitalization", 
        "TR.F.ReturnAvgTotAssetsPctTTM.date",
        "TR.F.ReturnAvgTotAssetsPctTTM",
        "TR.IPODate",                      
        "TR.DPSMean.date",
        "TR.DPSMean",
        "TR.F.BookValuePerShr.date", 
        "TR.F.BookValuePerShr",
        "TR.TRBCEconomicSector"
    ],
    parameters={
        'SDate': start_date.strftime('%Y-%m-%d'), 
        'EDate': end_date.strftime('%Y-%m-%d'), 
        'Frq': 'M'
    }
)

In [123]:
fundamentals.head(5)

,Instrument,Date,Company Market Capitalization,Date,"Return on Average Total Assets - %, TTM",IPO Date,Date,Dividend Per Share - Mean,Date,Book Value per Share,TRBC Economic Sector Name
0,AMRT.JK,2022-12-30,110039929505000.0,2022-09-30,9.245013,2009-01-15,2022-12-05,21.4885,2021-12-31,222.117054,Consumer Non-Cyclicals
1,AMRT.JK,2023-01-31,117514339811000.0,2022-09-30,9.245013,NaT,2022-12-05,21.4885,2021-12-31,222.117054,
2,AMRT.JK,2023-02-28,120421054930000.0,2022-09-30,9.245013,NaT,2022-12-05,21.4885,2021-12-31,222.117054,
3,AMRT.JK,2023-03-31,119590564896000.0,2022-12-31,10.005693,NaT,2022-12-05,21.4885,2022-12-31,270.238691,
4,AMRT.JK,2023-04-28,120421054930000.0,2022-12-31,10.005693,NaT,2022-12-16,28.40175,2022-12-31,270.238691,


In [124]:
# 1. Create a copy to protect the raw data
fund_clean = fundamentals.copy()

# 2. Identify columns by index to handle the duplicate 'Date' names
# Refinitiv Order: [0]Instrument, [1]Date1, [2]Cap, [3]Date2, [4]ROA, [5]IPO, [6]Date3, [7]Div, [8]Date4, [9]BVPS, [10]Sector
cols = fund_clean.columns.tolist()

# Define a new list of unique names based on the order you requested
new_names = [
    'Instrument', 'Date', 'Market_Cap', 'Date_Drop1', 'ROA', 
    'IPO_Date', 'Date_Drop2', 'DPS_Raw', 'Date_Drop3', 'BVPS', 'Sector'
]

# Assign the new names
fund_clean.columns = new_names

# 3. Drop the redundant date columns and keep only the first 'Date'
fund_clean = fund_clean.drop(columns=['Date_Drop1', 'Date_Drop2', 'Date_Drop3'])

# 4. Standardize the Master Date and Scale Market Cap
fund_clean['Date'] = pd.to_datetime(fund_clean['Date'])
fund_clean['Market_Cap_Bil'] = fund_clean['Market_Cap'] / 1_000_000_000

# 5. Create the Dividend Payer Dummy
fund_clean['Div_Payer'] = fund_clean['DPS_Raw'].apply(lambda x: 1 if pd.notnull(x) and x > 0 else 0)

# 6. Fix the Sector/IPO NaN issues (using the string fix from earlier)
fund_clean['Sector'] = fund_clean['Sector'].replace(r'^\s*$', pd.NA, regex=True).astype(object)
static_cols = ['IPO_Date', 'Sector']
for col in static_cols:
    fund_clean[col] = fund_clean.groupby('Instrument')[col].ffill().bfill()

print("Cleaned Panel Columns:")
print(fund_clean.columns.tolist())

Cleaned Panel Columns:
['Instrument', 'Date', 'Market_Cap', 'ROA', 'IPO_Date', 'DPS_Raw', 'BVPS', 'Sector', 'Market_Cap_Bil', 'Div_Payer']


In [125]:
fund_clean.head()

,Instrument,Date,Market_Cap,ROA,IPO_Date,DPS_Raw,BVPS,Sector,Market_Cap_Bil,Div_Payer
0,AMRT.JK,2022-12-30,110039929505000.0,9.245013,2009-01-15,21.4885,222.117054,Consumer Non-Cyclicals,110039.929505,1
1,AMRT.JK,2023-01-31,117514339811000.0,9.245013,2009-01-15,21.4885,222.117054,Consumer Non-Cyclicals,117514.339811,1
2,AMRT.JK,2023-02-28,120421054930000.0,9.245013,2009-01-15,21.4885,222.117054,Consumer Non-Cyclicals,120421.05493,1
3,AMRT.JK,2023-03-31,119590564896000.0,10.005693,2009-01-15,21.4885,270.238691,Consumer Non-Cyclicals,119590.564896,1
4,AMRT.JK,2023-04-28,120421054930000.0,10.005693,2009-01-15,28.40175,270.238691,Consumer Non-Cyclicals,120421.05493,1


In [132]:
# 5. Fill static fields across the 3-year panel
# Ensures Sector and IPO_Date propagate from the first row to all monthly rows
static_cols = ['IPO_Date', 'Sector']
for col in static_cols:
    fund_clean[col] = fund_clean.groupby('Instrument')[col].ffill().bfill()

# 6. Calculate Age & Create Dividend Dummy [cite: 48-49]
fund_clean['IPO_Date'] = pd.to_datetime(fund_clean['IPO_Date'])
fund_clean['Date'] = pd.to_datetime(fund_clean['Date'])

# Months since listing relative to the current date [cite: 48]
fund_clean['Months_Since_Listing'] = (pd.Timestamp.now() - fund_clean['IPO_Date']).dt.days // 30

# 1 if payer, 0 if speculative non-payer [cite: 49]
fund_clean['Div_Payer'] = fund_clean['DPS_Raw'].apply(lambda x: 1 if pd.notnull(x) and x > 0 else 0)

# DROP DPS_Raw: use axis=1 for columns and inplace=True to save changes
fund_clean.drop('DPS_Raw', axis=1, inplace=True)

# 7. Scale Market Cap for readability (IDR Billions)
fund_clean['Market_Cap_Bil'] = fund_clean['Market_Cap'] / 1_000_000_000

In [133]:
# 2. Check for missing data (NaNs)
print("--- Data Coverage (%) ---")
print(fund_clean.notnull().mean() * 100)

--- Data Coverage (%) ---
Instrument              100.000000
Date                     95.500000
Market_Cap               95.500000
ROA                      88.888889
IPO_Date                100.000000
BVPS                     99.611111
Sector                  100.000000
Market_Cap_Bil           95.500000
Div_Payer               100.000000
Months_Since_Listing    100.000000
dtype: float64


In [134]:
# Save the daily market data (Price, Volume, Volatility)
daily_long.to_csv('jci_test_daily_3yr.csv', index=False)

# Save the monthly fundamental characteristics (Size, ROA, Listing Age, Div_Payer)
fund_clean.to_csv('jci_test_fundamentals_3yr.csv', index=False)

print("Files saved successfully to the CodeBook file system.")

Files saved successfully to the CodeBook file system.
